<a href="https://colab.research.google.com/github/siriusGK/Codegk/blob/main/Task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
pip install pennylane


In [14]:
import pennylane as qml
import torch
import numpy as np
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# transform in the images
transform = transforms.Compose([
    transforms.Resize((4,4)),  # reduce dimension in gray scale with 4x4x1
    transforms.ToTensor()
])

mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transform)#importing the data from pytorch lib

loader = torch.utils.data.DataLoader(mnist, batch_size=2, shuffle=True)#now loading the 2 image data only

In [15]:
qubits_n = 4
dev = qml.device("default.qubit", wires=2*qubits_n + 1)#getiing quantum machine with feature default qubit

def quantum_embedding(image, weights, wires):

    # encode pixel values
    for i in range(qubits_n):
        qml.RY(image[i], wires=wires[i])

    # trainable layer
    for i in range(qubits_n):
        qml.RX(weights[i], wires=wires[i])

    # entanglement using cnot gates
    for i in range(qubits_n-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])

In [ ]:
@qml.qnode(dev, interface="torch")#providing the circuit wires for computation
def swap_test_circuit(img1, img2, weights):

    ancilla = 0
    wires1 = list(range(1, qubits_n+1))
    wires2 = list(range(qubits_n+1, 2*qubits_n+1))

    qml.Hadamard(wires=ancilla)

    quantum_embedding(img1, weights, wires1)# making 1st iamge quatum state on wire 1
    quantum_embedding(img2, weights, wires2)#making 2nd iamge quatum state on wire 2

    for i in range(qubits_n):
        qml.CSWAP(wires=[ancilla, wires1[i], wires2[i]])

    qml.Hadamard(wires=ancilla)

    return qml.expval(qml.PauliZ(ancilla))

In [ ]:
def fidelity(img1, img2, weights):# use to comparision that how dimilar is the 2 images
    result = swap_test_circuit(img1, img2, weights)# the result is between -1 and 1, we need to convert it to be between 0 and 1
    return (1 + result) / 2


In [18]:
def contrastive_loss(img1, img2, label1, label2, weights):

    F = fidelity(img1, img2, weights)

    if label1 == label2:
        loss = (1 - F)**2
    else:
        loss = (F)**2

    return loss

In [ ]:
weights = torch.randn(qubits_n, requires_grad=True)
opt = torch.optim.Adam([weights], lr=0.01)#using the adam optimizer

for epoch in range(5):

    for img, label in loader:

        img1 = img[0].flatten()[:qubits_n]#converting to 1D form (ex 28x28 ->784)
        img2 = img[1].flatten()[:qubits_n]

        label1 = label[0]
        label2 = label[1]

        loss = contrastive_loss(img1, img2, label1, label2, weights)

        opt.zero_grad()
        loss.backward()
        opt.step()

    print("epoch", epoch, "loss", loss.item())

epoch 0 loss 0.9984755665804166
epoch 1 loss 0.9982340822125724
epoch 2 loss 0.9939041906383569
